# PoliMillionaire - Multi-TF-IDF retrieval (SimpleWiki + KELM), no LLM

Analago a BM25 multi-index, ma usa TF-IDF come richiesto per il benchmark.

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [2]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [3]:
WIKI_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_tfidf.joblib"
KELM_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "kelm_500k_tfidf.joblib"

if not WIKI_INDEX_PATH.exists() or not KELM_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing one of the indexes")

wiki_index = load_retrieval_index(WIKI_INDEX_PATH)
print("Loaded SimpleWiki index:", WIKI_INDEX_PATH)

kelm_index = load_retrieval_index(KELM_INDEX_PATH)
print("Loaded KELM index:", KELM_INDEX_PATH)

indexes = [wiki_index, kelm_index]

Loaded SimpleWiki index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_tfidf.joblib
Loaded KELM index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\kelm_500k_tfidf.joblib


In [6]:
ATTEMPTS_PER_COMPETITION = 10
STRATEGY_NAME = "simplewiki_and_kelm_tfidf_no_llm"

rows = run_all_competitions(
    client=client,
    index=indexes,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_kelm_tfidf_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_and_kelm_tfidf_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=0 correct=True earned=100 latency=0.714395599992713
Q: Which of the following best describes the vocal range of Whitney Houston?
A: Mezzo-soprano
Top evidence: kelm_482207 :: Jane Bathori has a mezzo-soprano and soprano voice.

[simplewiki_and_kelm_tfidf_no_llm] comp=0 Entertainment attempt=1 q=2 level=0 chosen=1 correct=False earned=100 latency=0.8028529000002891
Q: What fundamental principle did Johnny Cash's 'Man in Black' song codify about his persona?
A: He was a rock and roll star
Top evidence: kelm_282820 :: Songs of Our Soil is part of Johnny Cash's discography of Johnny Cash albums. The discography of Johnny Cash, who is the author of Hymns by Johnny Cash, included Johnny Cash Sings Hank Williams.

[simplewiki_and_kelm_tfidf_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=1 correct=False earned=0 latency=0.7519415999995545
Q: In the context of film production, what does the term 'cas

In [5]:
summarize(rows)


Summary
0 Entertainment: rows=7, sessions=5, correct=2, row_acc=0.286, max_seen_level=0, avg_latency=0.733s
1 Ancient History and Politics: rows=9, sessions=5, correct=4, row_acc=0.444, max_seen_level=0, avg_latency=0.701s
2 Science and Nature: rows=5, sessions=5, correct=0, row_acc=0.000, max_seen_level=0, avg_latency=0.676s
3 Maths: rows=6, sessions=5, correct=1, row_acc=0.167, max_seen_level=0, avg_latency=0.639s
